In [4]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation, IGate, XGate, YGate, ZGate, UnitaryGate
from qiskit.quantum_info import Statevector, Operator

# 1. Define coefficients for B = 1.5*I - 0.5*X - 1.0*Y + 2.0*Z
coeffs = np.array([1.5, -0.5, -1.0, 2.0])
unitaries = [IGate(), XGate(), YGate(), ZGate()]

# Compute 1-norm lambda and positive square-root amplitudes
lam = np.sum(np.abs(coeffs))
prep_amps = np.sqrt(np.abs(coeffs) / lam)

# Create PREPARE and PREPARE^dagger gates
prep = StatePreparation(prep_amps, label="PREP")
prep_dag = prep.inverse()
prep_dag.label = "PREP†"

# Initialize 3-qubit circuit: q0, q1 = ancilla; q2 = system
qc = QuantumCircuit(3)

# --- 1. PREPARE ---
qc.append(prep, [0, 1])

# --- 2. SELECT (with signs absorbed directly into U_k) ---
for k, (c_k, gate) in enumerate(zip(coeffs, unitaries)):
    b_str = format(k, '02b')  # '00', '01', '10', '11'
    sign = np.sign(c_k)
    
    # Skip only if it is +Identity (since +I does nothing)
    if gate.name == 'id' and sign > 0:
        continue
        
    # Multiply the 2x2 unitary matrix by sign (+1 or -1)
    signed_matrix = sign * Operator(gate).data
    gate_label = f"{'-' if sign < 0 else ''}{gate.name.upper()}"
    
    signed_gate = UnitaryGate(signed_matrix, label=gate_label)
    qc.append(signed_gate.control(2, ctrl_state=b_str), [0, 1, 2])

# --- 3. PREPARE^dagger ---
qc.append(prep_dag, [0, 1])

# Now qc.draw() works cleanly!
print(qc.draw(fold=-1))

# --- 4. VERIFY STATEVECTOR & BLOCK ENCODING ---
psi_init = Statevector.from_label('000')  # |q2=0, q1=0, q0=0>
final_state = psi_init.evolve(qc)

# Extract branch where ancilla (q1, q0) == '00' (indices 0 and 4)
amps = final_state.data
branch_00 = np.array([amps[0], amps[4]])

# Classical comparison
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])

B = coeffs[0]*I + coeffs[1]*X + coeffs[2]*Y + coeffs[3]*Z
classical_B_over_lam = (B / lam) @ np.array([1, 0])

print("\nQuantum LCU branch |q1 q0 = 00> (unnormalized):")
print(np.round(branch_00, 6))

print("\nClassical (B / lambda)|0>:")
print(np.round(classical_B_over_lam, 6))

# --- 5. EXTRACT BLOCK-ENCODED MATRIX ---
U_full = Operator(qc).data
U_top_left_block = U_full[0::4, 0::4]

print("\nBlock-Encoded Matrix from Circuit U[0::4, 0::4]:")
print(np.round(U_top_left_block, 6))

print("\nExact Classical Matrix (B / lambda):")
print(np.round(B / lam, 6))

     ┌────────────────────────────────────────┐                 ┌─────────────────────────────────────────┐
q_0: ┤0                                       ├──■─────o─────■──┤0                                        ├
     │  PREP(0.54772,0.31623,0.44721,0.63246) │  │     │     │  │  PREP†(0.54772,0.31623,0.44721,0.63246) │
q_1: ┤1                                       ├──o─────■─────■──┤1                                        ├
     └────────────────────────────────────────┘┌─┴──┐┌─┴──┐┌─┴─┐└─────────────────────────────────────────┘
q_2: ──────────────────────────────────────────┤ -X ├┤ -Y ├┤ Z ├───────────────────────────────────────────
                                               └────┘└────┘└───┘                                           

Quantum LCU branch |q1 q0 = 00> (unnormalized):
[ 0.7-0.j  -0.1-0.2j]

Classical (B / lambda)|0>:
[ 0.7+0.j  -0.1-0.2j]

Block-Encoded Matrix from Circuit U[0::4, 0::4]:
[[ 0.7-0.j  -0.1+0.2j]
 [-0.1-0.2j -0.1+0.j ]]

Exact Classical Matrix (B